In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder 
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import mlflow
import joblib
import warnings
warnings.filterwarnings('ignore')

In [18]:
# charger le dataset de l'EDA
df_propre = pd.read_csv("dataset/df_eda_agritech.csv")

In [19]:
# configuration ML flow

URL_HF_SPACE = "https://gull1979-mlflow-tracking-ft.hf.space/" 
mlflow.set_tracking_uri(URL_HF_SPACE)
mlflow.set_experiment("AgriTech_Yield")



<Experiment: artifact_location='s3://fullstack-exo/mlflow-tracking/2', creation_time=1780586481799, experiment_id='2', last_update_time=1780586481799, lifecycle_stage='active', name='AgriTech_Yield', tags={}, trace_location=None, workspace='default'>

In [ ]:
# cibler les variables explicatives (X) et la target (Y)
colonnes_explicatives =[ x for x in df_propre.columns if x != 'Value']
X = df_propre.loc[:,colonnes_explicatives]
Y = df_propre.loc[:,['Value']]
Y



,Value
0,5217.4
1,2015.6
2,1500.0
3,1120.0
4,1488.0
...,...
28931,4026.8
28932,8127.9
28933,1464.6
28934,1827.0


In [ ]:
# répartition des données entre la partie entraînnement et la partie test
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=0)

In [ ]:
# lidentification des types de variables explicatives , soit numérique (type = float) soit catégorielle

numeric_feature = [x for x in colonnes_explicatives if df_propre.dtypes[x] == "float64"]
categorc_feature = [x for x in colonnes_explicatives if df_propre.dtypes[x] != "float64"]


In [ ]:
# premier bloc du pipeline, il s'agit de mettre à l'échelle les variables numériques
std_scaler_pipe =Pipeline(
    steps=[
        ('sdt_scal',StandardScaler())
    ]
)


In [ ]:
# deuxième bloc de pré-traitement des données, il s'agit ici de rendre numérique les variables catégorielles
    #TargetEncoder remplace chaque classe par la moyenne de la cible (pour cette classe)
        #exemple : France Pommes -> 30 000 à la place de france (disons que c'est la moyenne de tous les rendements pour la france pour toutes les récoltes)
            # pommes -> 3 000 (disons que c'est la moyenne de tous les rendements pour les pommes pour tous les pays)
cat_pipe = Pipeline(
    steps=[
        ('cat_encoder',TargetEncoder(smoothing=10)) #utilisation du paramètre smoothing afin de limiter le surapprentissage
    ]
)

In [ ]:
# pipeline 'preprocessing' final
# ce n'est actuellement pas le pipeline complet, il faudra ajouter le regressor/patern. 
cl_transformer = ColumnTransformer(transformers=
                                [
    ('num_transformer',std_scaler_pipe,numeric_feature),
    ('cat_transformer',cat_pipe,categorc_feature)
                                ]
)

### --------------------------------------------------------
### --- PARTIE RANDOMFOREST REGRESSOR --- BEGIN
### --------------------------------------------------------

# pipeline global avec le patern et le preprocessing
pipeline_global = Pipeline([
    ('preprocessor', cl_transformer),
    ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1))
])


param_grid = {
    "regressor__max_depth": [16,20,24], # Profondeur de l'arbre pour capter la complexité
    "regressor__max_features": ["sqrt",0.5,0.7], # ratio de features tirées au sort par nœud
    "regressor__min_samples_leaf": [2,3,4] # Taille minimale qui détermine la création d'une feuille (plus de noeud / sous-groupes)
}
#version utilise tout les coeurs CPU , n_jobs=-1)
obj_gridsearch = GridSearchCV(pipeline_global,param_grid=param_grid,cv=3, n_jobs=-1,scoring='r2')

### Fit et Métriques



with mlflow.start_run(run_name="Optimized_RandomForest_Pipeline"):
    # FIT
    obj_gridsearch.fit(X_train,Y_train)

    # 1. On récupère le meilleur estimateur trouvé par la GridSearch 
    le_modele = obj_gridsearch.best_estimator_

    # 3. Prédictions
    y_pred = le_modele.predict(X_test)

    # Calcul de chaque métrique
    mae = mean_absolute_error(Y_test, y_pred)
    mse = mean_squared_error(Y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(Y_test, y_pred)
    score_train = le_modele.score(X_train, Y_train)
    score_test = le_modele.score(X_test, Y_test)
    ecart_overfitting = score_train - score_test

    # metric MLFLOW
    mlflow.log_metric("R2_test", r2)
    mlflow.log_metric("MAE_test", mae)
    mlflow.log_metric("MSE_test", mse)
    mlflow.log_metric("RMSE_test", rmse)
    mlflow.log_metric("R2_train", score_train)
    mlflow.log_metric("Ecart_Overfitting", ecart_overfitting)

    print('Métrique\t\t\t\tvaleur')
    print("R² (Coefficient de détermination) \t{}".format(round(r2,2)))
    print("MAE (Mean Absolute Error) \t\t{}".format(round(mae,2)))
    print("MSE (Mean Squared Error) \t\t{}".format(round(mse,2)))
    print("RMSE (Root Mean Squared Error) \t\t{}".format(round(rmse,2)))
    print("Niveau d'overfitting par l'écart de score entre train et test \t{}".format(ecart_overfitting))
    print("Meilleurs hyperparamètres trouvés :", obj_gridsearch.best_params_)


## commentaire optimisation paramètre randomforestregressor
'''
Run 1 :
param_grid = {
    "regressor__max_depth": [10, 15, 20],
    "regressor__max_features": ["sqrt", 0.5],
    "regressor__min_samples_leaf": [1, 2]
}

max_depth
20
min_samples_leaf
2

r2 test : 0.8916266855576203
7.1% overfitting

Run 2 :
param_grid = {
    "regressor__max_depth": [8,10,12],
    "regressor__max_features": ["sqrt"],
    "regressor__min_samples_leaf": [5,10,15]
}

max_depth
12
min_samples_leaf
5

r2 test : 0.8375386136619247
3.5% overfitting

Run 3 :
param_grid = {
    "regressor__max_depth": [16,20,24],
    "regressor__max_features": ["sqrt",0.5,0.7],
    "regressor__min_samples_leaf": [2,3,4]
}

max_depth
24
min_samples_leaf
2
max_features
0.7

r2 test : 0.91
6.5% overfitting

'''
## Le Run 3 offre le meilleur compromis entre un R² de test de 91% pour un overfitting de 6.5%

le_modele

### --------------------------------------------------------
### --- PARTIE RANDOMFOREST REGRESSOR --- END
### --------------------------------------------------------

### --------------------------------------------------------
### --- PARTIE XGBoost REGRESSOR --- BEGIN
### --------------------------------------------------------

# pipeline global avec le patern et le preprocessing
pipeline_global = Pipeline([
    ('preprocessor', cl_transformer),
    ('regressor', XGBRegressor(random_state=42, n_jobs=-1, objective='reg:squarederror')) #fonction de coût MSE
])


param_grid = {
    "regressor__n_estimators": [300 ], # Nombre d'arbres construits séquentiellement
    "regressor__max_depth": [8], # Profondeur de l'arbre pour capter la complexité
    "regressor__learning_rate": [0.1], # Vitesse de correction (poids accordé à chaque nouvel arbre pour corriger l'erreur du précédent)
    "regressor__colsample_bytree": [0.7] # ratio de features tirées au sort pour la totalité de l'arbre
}
#version utilise tout les coeurs CPU , n_jobs=-1)
obj_gridsearch = GridSearchCV(pipeline_global,param_grid=param_grid,cv=3, n_jobs=-1,scoring='r2')



with mlflow.start_run(run_name="XGBRegressor_Optimized"):
    # FIT
    obj_gridsearch.fit(X_train,Y_train)

    # 1. On récupère le meilleur estimateur trouvé par la GridSearch 
    le_modele = obj_gridsearch.best_estimator_

    # 3. Prédictions
    y_pred = le_modele.predict(X_test)

    # Calcul de chaque métrique
    mae = mean_absolute_error(Y_test, y_pred)
    mse = mean_squared_error(Y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(Y_test, y_pred)
    score_train = le_modele.score(X_train, Y_train)
    score_test = le_modele.score(X_test, Y_test)
    ecart_overfitting = score_train - score_test


    # metric MLFLOW
    mlflow.log_metric("R2_test", r2)
    mlflow.log_metric("MAE_test", mae)
    mlflow.log_metric("MSE_test", mse)
    mlflow.log_metric("RMSE_test", rmse)
    mlflow.log_metric("R2_train", score_train)
    mlflow.log_metric("Ecart_Overfitting", ecart_overfitting)
   

    print('Métrique\t\t\t\tvaleur')
    print("R² (Coefficient de détermination) \t{}".format(round(r2,2)))
    print("MAE (Mean Absolute Error) \t\t{}".format(round(mae,2)))
    print("MSE (Mean Squared Error) \t\t{}".format(round(mse,2)))
    print("RMSE (Root Mean Squared Error) \t\t{}".format(round(rmse,2)))
    print("Niveau d'overfitting par l'écart de score entre train et test \t{}".format(ecart_overfitting))
    print("Meilleurs hyperparamètres trouvés :", obj_gridsearch.best_params_)


In [ ]:
## commentaire optimisation paramètre XGBRegressor
'''
run  1 :
param_grid = {
    "regressor__n_estimators": [100, 200, 300], 
    "regressor__max_depth": [4, 6, 8], 
    "regressor__learning_rate": [0.03, 0.07, 0.1], 
    "regressor__colsample_bytree": [0.7, 0.9] 
}

Meilleurs hyperparamètres trouvés : {'regressor__colsample_bytree': 0.7, 'regressor__learning_rate': 0.1, 'regressor__max_depth': 8, 'regressor__n_estimators': 300}

r2 test : 0.92
5.6% overfitting

run 2 :
param_grid = {
    "regressor__n_estimators": [300 , 500 , 800], 
    "regressor__max_depth": [8 , 10], 
    "regressor__learning_rate": [0.05, 0.1, 0.15],
    "regressor__colsample_bytree": [0.6, 0.7, 0.8] 
}

Meilleurs hyperparamètres trouvés : {'regressor__colsample_bytree': 0.7, 'regressor__learning_rate': 0.1, 'regressor__max_depth': 8, 'regressor__n_estimators': 800}

r2 test : 0.92
6.7% overfitting


xgboost_final = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1
)


'''
## Le Run 1 offre le meilleur compromis entre un R² de test de 92% pour un overfitting de 5.6% , le run 2 avec des limites de paramètres élargies a augmenter l'overfitting et déséquilibré le rapport biais-variance



### --------------------------------------------------------
### --- PARTIE XGBoost REGRESSOR --- END
### --------------------------------------------------------

### --------------------------------------------------------
### --- PARTIE lightgbm REGRESSOR --- BEGIN
### --------------------------------------------------------

# pipeline global avec le patern et le preprocessing
pipeline_global = Pipeline([
    ('preprocessor', cl_transformer),
    ('regressor', LGBMRegressor(random_state=42, n_jobs=1, objective='regression', subsample_freq=1,verbose=-1)) #fonction de coût MSE
])


param_grid = {
    "regressor__n_estimators": [500], # Nombre d'arbres construits séquentiellement
    "regressor__num_leaves": [127], # défini le nombre total de feuilles du modèle
    "regressor__max_depth": [ -1], # Profondeur de l'arbre pour capter la complexité (-1 pas de limite)
    "regressor__m": [ 0.1], # Vitesse de correction (poids accordé à chaque nouvel arbre pour corriger l'erreur du précédent)
    "regressor__subsample": [0.9] # ratio d'observations tirées au sort pour la totalité de l'arbre
}

#version utilise tout les coeurs CPU , n_jobs=-1)
obj_gridsearch = GridSearchCV(pipeline_global,param_grid=param_grid,cv=3, n_jobs=-1,scoring='r2')


with mlflow.start_run(run_name="LightGBM_Optimized"):
    # FIT
    obj_gridsearch.fit(X_train,Y_train)

    # 1. On récupère le meilleur estimateur trouvé par la GridSearch 
    le_modele = obj_gridsearch.best_estimator_

    # 3. Prédictions
    y_pred = le_modele.predict(X_test)

    # Calcul de chaque métrique
    mae = mean_absolute_error(Y_test, y_pred)
    mse = mean_squared_error(Y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(Y_test, y_pred)
    score_train = le_modele.score(X_train, Y_train)
    score_test = le_modele.score(X_test, Y_test)
    ecart_overfitting = score_train - score_test


    # metric MLFLOW
    mlflow.log_metric("R2_test", r2)
    mlflow.log_metric("MAE_test", mae)
    mlflow.log_metric("MSE_test", mse)
    mlflow.log_metric("RMSE_test", rmse)
    mlflow.log_metric("R2_train", score_train)
    mlflow.log_metric("Ecart_Overfitting", ecart_overfitting)
   

    print('Métrique\t\t\t\tvaleur')
    print("R² (Coefficient de détermination) \t{}".format(round(r2,2)))
    print("MAE (Mean Absolute Error) \t\t{}".format(round(mae,2)))
    print("MSE (Mean Squared Error) \t\t{}".format(round(mse,2)))
    print("RMSE (Root Mean Squared Error) \t\t{}".format(round(rmse,2)))
    print("Niveau d'overfitting par l'écart de score entre train et test \t{}".format(ecart_overfitting))
    print("Meilleurs hyperparamètres trouvés :", obj_gridsearch.best_params_)


## commentaire optimisation paramètre LightGBM
'''
run  1 :
param_grid = {
    "regressor__n_estimators": [100, 300, 500], 
    "regressor__num_leaves": [31, 63, 127], 
    "regressor__max_depth": [6, 8, -1],
    "regressor__learning_rate": [0.03, 0.07, 0.1],
    "regressor__subsample": [0.7, 0.9] 
}

Meilleurs hyperparamètres trouvés : {'regressor__learning_rate': 0.1, 'regressor__max_depth': -1, 'regressor__n_estimators': 500, 'regressor__num_leaves': 127, 'regressor__subsample': 0.9}

r2 test : 0.92
6.5% overfitting

run 2 :
param_grid = {
    "regressor__n_estimators": [500, 700, 1000], 
    "regressor__num_leaves": [127, 150, 200], 
    "regressor__max_depth": [ -1], 
    "regressor__learning_rate": [0.07, 0.1, 0.15], 
    "regressor__subsample": [0.8, 0.9] 
}

Meilleurs hyperparamètres trouvés : {'regressor__learning_rate': 0.07, 'regressor__max_depth': -1, 'regressor__n_estimators': 700, 'regressor__num_leaves': 200, 'regressor__subsample': 0.9}

r2 test : 0.92
7% overfitting


LightGBM_final = LightGBM(
    n_estimators=500,
    max_depth=-1,
    learning_rate=0.1,
    num_leaves=127,
    subsample=0.9,
    objective='regression',
    random_state=42,
    subsample_freq=1,
    n_jobs=1,
    verbose=-1
)


'''
## Le Run 1 offre le meilleur compromis entre un R² de test de 92% pour un overfitting de 6.5% , le run 2 avec des limites de paramètres élargies a augmenter légèrement l'overfitting 



### --------------------------------------------------------
### --- PARTIE lightgbm REGRESSOR --- END
### --------------------------------------------------------

### --------------------------------------------------------
### --- PARTIE CatBoost REGRESSOR --- BEGIN
### --------------------------------------------------------

# pipeline global avec le patern et le preprocessing
pipeline_global = Pipeline([
    ('preprocessor', cl_transformer),
    ('regressor', CatBoostRegressor(random_state=42,thread_count=1, loss_function='RMSE',verbose=False)) #fonction de coût RMSE
])

param_grid = {
    "regressor__iterations": [500, 1000, 1500], # Nombre d'arbres construits séquentiellement
    "regressor__depth": [6, 8, 10], # Profondeur de l'arbre pour capter la complexité 
    "regressor__learning_rate": [0.01, 0.05, 0.1], # Vitesse de correction (poids accordé à chaque nouvel arbre pour corriger l'erreur du précédent)
    "regressor__l2_leaf_reg": [2, 5, 10] # Coefficient de régularisation L2  pour limiter l'overfitting
}

#version utilise tout les coeurs CPU , n_jobs=-1)
obj_gridsearch = GridSearchCV(pipeline_global,param_grid=param_grid,cv=3, n_jobs=-1,scoring='r2')


with mlflow.start_run(run_name="CatBoost_Optimized"):
    # FIT
    obj_gridsearch.fit(X_train,Y_train)

    # 1. On récupère le meilleur estimateur trouvé par la GridSearch 
    le_modele = obj_gridsearch.best_estimator_

    # 3. Prédictions
    y_pred = le_modele.predict(X_test)

    # Calcul de chaque métrique
    mae = mean_absolute_error(Y_test, y_pred)
    mse = mean_squared_error(Y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(Y_test, y_pred)
    score_train = le_modele.score(X_train, Y_train)
    score_test = le_modele.score(X_test, Y_test)
    ecart_overfitting = score_train - score_test


    # metric MLFLOW
    mlflow.log_metric("R2_test", r2)
    mlflow.log_metric("MAE_test", mae)
    mlflow.log_metric("MSE_test", mse)
    mlflow.log_metric("RMSE_test", rmse)
    mlflow.log_metric("R2_train", score_train)
    mlflow.log_metric("Ecart_Overfitting", ecart_overfitting)
   

    print('Métrique\t\t\t\tvaleur')
    print("R² (Coefficient de détermination) \t{}".format(round(r2,2)))
    print("MAE (Mean Absolute Error) \t\t{}".format(round(mae,2)))
    print("MSE (Mean Squared Error) \t\t{}".format(round(mse,2)))
    print("RMSE (Root Mean Squared Error) \t\t{}".format(round(rmse,2)))
    print("Niveau d'overfitting par l'écart de score entre train et test \t{}".format(ecart_overfitting))
    print("Meilleurs hyperparamètres trouvés :", obj_gridsearch.best_params_)


## commentaire optimisation paramètre CatBoost
'''
run  1 :
param_grid = {
    "regressor__iterations": [100, 300, 500], 
    "regressor__depth": [4, 6, 8], 
    "regressor__learning_rate": [0.03, 0.07, 0.1], 
    "regressor__l2_leaf_reg": [1, 3, 5] 
}

Meilleurs hyperparamètres trouvés : {'regressor__depth': 8, 'regressor__iterations': 500, 'regressor__l2_leaf_reg': 1, 'regressor__learning_rate': 0.1}

r2 test : 0.9
3.6% overfitting




run 2 :
param_grid = {
    "regressor__iterations": [500, 1000, 1500],
    "regressor__depth": [6, 8, 10], 
    "regressor__learning_rate": [0.01, 0.05, 0.1], 
    "regressor__l2_leaf_reg": [2, 5, 10] 
}

Meilleurs hyperparamètres trouvés : {'regressor__depth': 10, 'regressor__iterations': 1500, 'regressor__l2_leaf_reg': 2, 'regressor__learning_rate': 0.1}

r2 test : 0.927
5% overfitting



'''
## Le Run 2 offre le meilleur compromis entre un R² de test de 92% pour un overfitting de 5% .



### --------------------------------------------------------
### --- PARTIE CatBoost REGRESSOR --- END
### --------------------------------------------------------

### --------------------------------------------------------
### --- CatBoost Production --- BEGIN
### --------------------------------------------------------

In [ ]:
# pipeline global avec le patern et le preprocessing
CatBoost_pipeline_global = Pipeline([
    ('preprocessor', cl_transformer),
    ('regressor', CatBoostRegressor(
        random_state=42, # 
        thread_count=-1, # utiliser tous mes coeurs du processeur
        loss_function='RMSE', # type de fonction de coût
        verbose=False, # limiter la quantité d'affichage dans la console
        iterations=1500, # Nombre d'arbres construits séquentiellement
        l2_leaf_reg=2, # Coefficient de régularisation L2  pour limiter l'overfitting
        learning_rate=0.1, # Vitesse de correction (poids accordé à chaque nouvel arbre pour corriger l'erreur du précédent)
        depth=10 # Profondeur de l'arbre pour capter la complexité 
        ))
])


In [29]:
CatBoost_pipeline_global.fit(X_train,Y_train)
joblib.dump(CatBoost_pipeline_global, "CatBoost_Prod.joblib")



['CatBoost_Prod.joblib']

### --------------------------------------------------------
### --- CatBoost Production --- END
### --------------------------------------------------------